In [ ]:
# Part 1: Content-Based Filtering :- This method recommends movies based on similarity of genres using TF-IDF and cosine similarity.
# a: Item-based similarity
# b: User profile-based recommendation

import pandas as pd

movies = pd.read_csv("ml-latest/movies.csv")
ratings = pd.read_csv("ml-latest/ratings.csv")
tags = pd.read_csv("ml-latest/tags.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

movies.head()

Movies: (86537, 3)
Ratings: (33832162, 4)
Tags: (2328315, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
#checkinh null values

movies.isnull().sum()
ratings.isnull().sum()
tags.isnull().sum()

userId        0
movieId       0
tag          17
timestamp     0
dtype: int64

In [4]:
#cleaning for ML use
movies['genres'] = movies['genres'].str.replace('|', ' ')
movies['genres'].head()


0    Adventure Animation Children Comedy Fantasy
1                     Adventure Children Fantasy
2                                 Comedy Romance
3                           Comedy Drama Romance
4                                         Comedy
Name: genres, dtype: str

In [6]:
movies['genres'] = movies['genres'].replace('(no genres listed)', '')
movies['content'] = movies['genres']
movies[['title', 'content']].head()

,title,content
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy Romance
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['content'])

print(tfidf_matrix.shape)

# Each movie is a vector and each genre is a feature. Output  will be a matrix of movies and features
# after code run we get 86537 movies, 21 unique genre features, Each movie is now a vector in 21-dimensional space.



(86537, 21)


In [ ]:
#cosine similarity
#from sklearn.metrics.pairwise import cosine_similarity
#cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
#print(cosine_sim.shape)
# running a 86573x86573 similarity asking to allocate 56gb ram, so starting with one movie at a time.
from sklearn.metrics.pairwise import linear_kernel
# Using linear kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix[:1])

#creating index mapping
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()
#recommendation function
from sklearn.metrics.pairwise import linear_kernel

def recommend(movie_title, top_n=5):
    idx = indices[movie_title]
    
    # computing similarity only for this movie
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    # top similar movies
    sim_indices = sim_scores.argsort()[-top_n-1:-1][::-1]
    
    return movies['title'].iloc[sim_indices]

recommend("Toy Story (1995)")

61975                    UglyDolls (2019)
85344       Mavka: The Forest Song (2023)
11606              Shrek the Third (2007)
72622    Legends of Valhalla: Thor (2011)
79535        Casper's Scare School (2006)
Name: title, dtype: str

In [13]:
data = ratings.merge(movies, on='movieId')
data.head()
user_id = 1
user_data = data[data['userId'] == user_id]
user_data.head()
# mapping creation
movie_id_to_index = pd.Series(movies.index, index=movies['movieId'])

# user movies to indices mapping
user_indices = user_data['movieId'].map(movie_id_to_index)
user_indices = user_indices.dropna().astype(int)
import numpy as np

user_profile = np.zeros(tfidf_matrix.shape[1])

for idx, rating in zip(user_indices, user_data['rating']):
    user_profile += tfidf_matrix[idx].toarray().flatten() * rating

# normalize
user_profile = user_profile / user_data['rating'].sum()
#computing similarity with all movies
from sklearn.metrics.pairwise import linear_kernel

user_profile = user_profile.reshape(1, -1)

sim_scores = linear_kernel(user_profile, tfidf_matrix).flatten()
#getting recommendation
sim_indices = sim_scores.argsort()[-10:][::-1]

recommended_movies = movies.iloc[sim_indices][['title']]
recommended_movies.head(5)

#recommender function
def recommend_user(user_id, top_n=5):
    user_data = data[data['userId'] == user_id]
    
    user_indices = user_data['movieId'].map(movie_id_to_index)
    
    user_profile = np.zeros(tfidf_matrix.shape[1])
    
    for idx, rating in zip(user_indices, user_data['rating']):
        user_profile += tfidf_matrix[idx].toarray().flatten() * rating
    
    user_profile = user_profile / user_data['rating'].sum()
    
    user_profile = user_profile.reshape(1, -1)
    
    sim_scores = linear_kernel(user_profile, tfidf_matrix).flatten()
    
    sim_indices = sim_scores.argsort()[-top_n:][::-1]
    
    return movies.iloc[sim_indices][['title']]

recommend_user(1)

,title
61461,Gone with the Bullets (2014)
14332,Legend of the Red Dragon (a.k.a. New Legend of...
48149,The Babymoon (2017)
10501,Casanova (2005)
76181,Pretty Guardian Sailor Moon Eternal The Movie ...


In [ ]:
# Part 2: Collaborative Filtering: This method recommends based on similar users using user-item matrix and cosine similarity.
# Task 1: User-User Collaborative Filtering
# Rows = users, Columns = movies, Values = ratings
# Checking the memory required

user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')
user_movie_matrix.head()



MemoryError: Unable to allocate 258. MiB for an array with shape (33832162,) and data type int64

In [19]:
# Reducing the data so that less memory can be used 
sample_users = ratings['userId'].unique()[:500]   # taking first 500 users
ratings_small = ratings[ratings['userId'].isin(sample_users)]
#creating matrix on small data
user_movie_matrix = ratings_small.pivot_table(index='userId', columns='movieId',values='rating')

#there could be NA values so dropping them
user_movie_matrix_filled = user_movie_matrix.fillna(0)

#user similarity
from sklearn.metrics.pairwise import cosine_similarity
def get_similar_users(user_id, top_n=5):
    user_vector = user_movie_matrix_filled.loc[user_id].values.reshape(1, -1)
    
    similarities = cosine_similarity(user_vector, user_movie_matrix_filled).flatten()
    
    sim_series = pd.Series(similarities, index=user_movie_matrix_filled.index)
    
    sim_series = sim_series.drop(user_id)  # remove self
    
    return sim_series.sort_values(ascending=False).head(top_n)

get_similar_users(1)

userId
467    0.316512
480    0.308960
299    0.295707
6      0.277858
331    0.275508
dtype: float64

In [21]:
# item - item similarity colaborative filtering (movie vs movie)
#row = movies, Columns = users
movie_user_matrix = user_movie_matrix.T
movie_user_matrix_filled = movie_user_matrix.fillna(0)
#similarity funstion
from sklearn.metrics.pairwise import cosine_similarity

def get_similar_movies(movie_id, top_n=5):
    if movie_id not in movie_user_matrix_filled.index:
        return "Movie not found"
    
    movie_vector = movie_user_matrix_filled.loc[movie_id].values.reshape(1, -1)
    similarities = cosine_similarity(movie_vector, movie_user_matrix_filled).flatten()
    sim_series = pd.Series(similarities, index=movie_user_matrix_filled.index)
    sim_series = sim_series.drop(movie_id)
    return sim_series.sort_values(ascending=False).head(top_n)
get_similar_movies(1)
def recommend_similar_movies(movie_id, top_n=5):
    sim_movies = get_similar_movies(movie_id, top_n)
    return movies[movies['movieId'].isin(sim_movies.index)][['title']]
recommend_similar_movies(1)

,title
257,Star Wars: Episode IV - A New Hope (1977)
351,Forrest Gump (1994)
475,Jurassic Park (1993)
1237,Back to the Future (1985)
3021,Toy Story 2 (1999)


In [ ]:
# Part 3: Matrix Factorization (SVD): SVD decomposes the matrix to capture latent features and predict missing ratings.

# Creating user - item matrix
user_movie_matrix = ratings_small.pivot_table(index='userId', columns='movieId', values='rating')
#Filling missing values
R = user_movie_matrix.fillna(0).values
import numpy as np
U, sigma, Vt = np.linalg.svd(R, full_matrices=False)
#Reducing dimentions
k = 50
U_k = U[:, :k]
sigma_k = np.diag(sigma[:k])
Vt_k = Vt[:k, :]
# Reconstructing matrix
R_pred = np.dot(np.dot(U_k, sigma_k), Vt_k)
#Converting back to dataframe
pred_df = pd.DataFrame(R_pred, index=user_movie_matrix.index, columns=user_movie_matrix.columns)
# Recommendation Function
def recommend_svd(user_id, top_n=5):
    user_ratings = user_movie_matrix.loc[user_id]
    predicted_ratings = pred_df.loc[user_id]
    unseen_movies = predicted_ratings[user_ratings.isna()]
    recommendations = unseen_movies.sort_values(ascending=False).head(top_n)
    return movies[movies['movieId'].isin(recommendations.index)][['title']]
recommend_svd(1)

#RMSE
from sklearn.metrics import mean_squared_error
import numpy as np

actual = user_movie_matrix.values.flatten()
predicted = pred_df.values.flatten()
mask = ~np.isnan(actual)
rmse = np.sqrt(mean_squared_error(actual[mask], predicted[mask]))

print("RMSE:", rmse)

#Precision@k
def precision_at_k(user_id, k=5):
    predicted_ratings = pred_df.loc[user_id]
    top_k = predicted_ratings.sort_values(ascending=False).head(k).index
    user_actual = ratings_small[
    (ratings_small['userId'] == user_id) & (ratings_small['rating'] >= 3)]['movieId']
    relevant = set(top_k).intersection(set(user_actual))
    return len(relevant) / k

#Recall@k
def recall_at_k(user_id, k=5):
    predicted_ratings = pred_df.loc[user_id]
    top_k = predicted_ratings.sort_values(ascending=False).head(k).index
    user_actual = ratings_small[
    (ratings_small['userId'] == user_id) & (ratings_small['rating'] >= 3)]['movieId']
    relevant = set(top_k).intersection(set(user_actual))
    return len(relevant) / len(user_actual) if len(user_actual) > 0 else 0

print("Precision@5:", precision_at_k(1))
print("Recall@5:", recall_at_k(1))

# Evaluating on multiple users (IMPORTANT)

users = ratings_small['userId'].unique()[:50]
precisions = []
recalls = []
for u in users:
    try:
        precisions.append(precision_at_k(u))
        recalls.append(recall_at_k(u))
    except:
        continue

print("Average Precision@5:", np.mean(precisions))
print("Average Recall@5:", np.mean(recalls))

RMSE: 1.8992389569950263
Precision@5: 1.0
Recall@5: 0.08333333333333333
Average Precision@5: 0.748
Average Recall@5: 0.15337733591482672


In [ ]:
# Part 4: Hybrid Recommendation: Combines content-based and collaborative filtering for better accuracy.
# Already have tfidf_matrix, linear_kernel, indices from part 1 :- Content-Based Filtering
# pred_df from collab score (SVD)

# Combining both scores
from sklearn.metrics.pairwise import linear_kernel

def content_score(movie_title):
    idx = indices[movie_title]
    sim_scores = linear_kernel(tfidf_matrix[idx], tfidf_matrix).flatten()
    return sim_scores
def hybrid_recommendation(user_id, movie_title, top_n=5, alpha=0.5):
    
    # Content score
    content_scores = content_score(movie_title)
    content_scores = pd.Series(content_scores, index=movies.index)
    
    # Collaborative score
    collab_scores = pred_df.loc[user_id]
    
    # Normalizing both
    content_scores = (content_scores - content_scores.min()) / (content_scores.max() - content_scores.min())
    collab_scores = (collab_scores - collab_scores.min()) / (collab_scores.max() - collab_scores.min())
    
    # Combining
    hybrid_scores = alpha * content_scores + (1 - alpha) * collab_scores
    
    # Top recommendations
    top_movies = hybrid_scores.sort_values(ascending=False).head(top_n).index
    
    return movies.iloc[top_movies][['title']]

hybrid_recommendation(1, "Toy Story (1995)")

,title
1,Jumanji (1995)
480,Last Action Hero (1993)
4262,Atlantis: The Lost Empire (2001)
78081,My Little Pony: A New Generation (2021)
3654,"Adventures of Rocky and Bullwinkle, The (2000)"


In [38]:
# Part 5: Learning-Based Recommender

import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

In [51]:
#Movie feature
#Limiting data size and ratings due to memory constraints
# Reduce dataset size
sample_movies = movies.sample(5000, random_state=42)
ratings_small_nn = ratings[ratings['movieId'].isin(sample_movies['movieId'])]
movies = sample_movies
ratings = ratings_small_nn

# Extracting genres
movies['genres'] = movies['genres'].apply(lambda x: x if isinstance(x, list) else str(x).split('|'))

mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(mlb.fit_transform(movies['genres']), columns=mlb.classes_, index=movies.index)

# Extracting year
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['year'] = movies['year'].fillna(movies['year'].mean())

# Avg rating per movie
if 'avg_rating' in movies.columns:
    movies = movies.drop(columns=['avg_rating'])

avg_rating = ratings.groupby('movieId')['rating'].mean().reset_index()
avg_rating.rename(columns={'rating': 'avg_rating'}, inplace=True)
movies = movies.merge(avg_rating, on='movieId', how='left')
movies['avg_rating'] = movies['avg_rating'].fillna(movies['avg_rating'].mean())

# Final movie features
# Reducing genre dimensions (taking top 20 most frequent genres) due to memory contraints for training.
top_genres = genre_matrix.sum().sort_values(ascending=False).head(20).index
genre_reduced = genre_matrix[top_genres]
movie_features = pd.concat([genre_reduced, movies[['year', 'avg_rating']]], axis=1)

movie_features.head()


,Drama,Documentary,Comedy,,Comedy Drama,Horror,Drama Romance,Comedy Romance,Comedy Drama Romance,Thriller,...,Animation,Horror Thriller,Western,Drama War,Action,Romance,Sci-Fi,Action Drama,year,avg_rating
1501,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2005.0,2.250000
2586,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2017.0,3.250000
2653,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1914.0,2.125000
1055,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1972.0,2.866667
705,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1958.0,3.346154


In [52]:
# User features (avg rating per genre)
data = ratings.merge(movies[['movieId', 'genres']], on='movieId')
data = data.explode('genres')
user_genre = data.groupby(['userId', 'genres'])['rating'].mean().unstack().fillna(0)

# Aligning data
common_users = user_genre.index.intersection(ratings['userId'])
common_movies = movies['movieId']
train_data = ratings[(ratings['userId'].isin(common_users)) & (ratings['movieId'].isin(common_movies))]

# Preparing inputs
movie_features_indexed = movie_features.copy()
movie_features_indexed['movieId'] = movies['movieId'].values
movie_features_indexed.set_index('movieId', inplace=True)
X_movie = movie_features_indexed.loc[train_data['movieId']].values

"""The neural network captures complex relationships between user preferences and movie features.
Unlike TF-IDF which only considers similarity, the neural model learns deeper interactions between genres, ratings, and user behavior.
Hence, it can provide more personalized recommendations."""

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate

# Inputs
user_input = Input(shape=(X_user.shape[1],))
movie_input = Input(shape=(X_movie.shape[1],))

# User embedding
u = Dense(32, activation='relu')(user_input)
u = Dense(16)(u)

# Movie embedding
m = Dense(32, activation='relu')(movie_input)
m = Dense(16)(m)

# Combining
x = Concatenate()([u, m])
x = Dense(32, activation='relu')(x)

output = Dense(1)(x)
model = Model([user_input, movie_input], output)
model.compile(optimizer='adam', loss='mse')
model.summary()
# Reducing training data due to memory constraints
# Reduce train size
train_data = train_data.sample(10000, random_state=42)

# Reduce genre features
top_genres = genre_matrix.sum().sort_values(ascending=False).head(20).index
genre_reduced = genre_matrix[top_genres]
movie_features = pd.concat([genre_reduced, movies[['year', 'avg_rating']]], axis=1)

# Index mapping
movie_features_indexed = movie_features.copy()
movie_features_indexed['movieId'] = movies['movieId'].values
movie_features_indexed.set_index('movieId', inplace=True)

# Final inputs
X_user = user_genre.loc[train_data['userId']].values.astype('float32')
X_movie = movie_features_indexed.loc[train_data['movieId']].values.astype('float32')
y = train_data['rating'].values.astype('float32')

print("Neural RMSE:", rmse_nn)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
!pip install shap
import shap
import numpy as np

# Example: movie_features already created
X = movie_features.values

# Train simple model (surrogate)
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=50)
model.fit(X, movies['avg_rating'])

# SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X[:1])

# Plot
shap.summary_plot(shap_values, movie_features.iloc[:1])



: 